# MandiaCompact — étape 8 terminée

**Deux entraînements de 400 pas sur NVIDIA L4**, même budget de 1 048 745 tokens par bras. Sélection sur validation : **20,86 M, checkpoint 100**. Les deux variantes restent expérimentales.

| Mesure | Parent | 13,78 M | 20,86 M |
|---|---:|---:|---:|
| Réponses exactes quotidien /40 | 6 | 15 | 16 |
| chrF++ quotidien | 32,59 | 48,39 | 49,51 |
| Réponses exactes Bible /8 | 0 | 0 | 0 |
| chrF++ Bible | 16,31 | 19,63 | 19,19 |
| Réponses simples entraînées | 14/14 | 14/14 | 14/14 |

**Limites :** les 40 cibles du quotidien sont déjà connues ; certaines anciennes pertes régressent. Les sorties bibliques présentent encore des erreurs de sens. Un seul exemple exact supplémentaire pour 51 % de paramètres supplémentaires ne démontre pas une meilleure efficacité. Le mélange du pilote biblique et de la continuation ne permet pas d'attribuer à la Bible le gain sur le quotidien.

**Sauvegarde Drive vérifiée : 6 fichiers de poids/checkpoints, 693 Mo**, avec AdamW, RNG et compteurs pour reprendre à 400. Les meilleurs poids à 100 et les états de reprise à 400 sont distincts. Aucun checkpoint entraîné téléchargé sur la machine locale.

## Lectures appliquées

Huit articles et notes dans le dossier de cette étape. Pistes directement testées : [X-Instruction](https://aclanthology.org/2024.findings-acl.30/) pour les questions autour de cibles attestées ; [Net2Net](https://arxiv.org/abs/1511.05641) pour l'idée d'élargissement préservant la fonction. [Optimizing Transformer](https://aclanthology.org/2020.coling-main.304/) et [Efficient Architectures](https://aclanthology.org/2025.lowresnlp-1.6/) motivent une comparaison mesurée ; leurs scores ne sont pas des résultats Mandia.

## Reproduction et origine des sorties

Le code de la cellule ci-dessous a réellement été exécuté dans la session privée du notebook étape 6 le 14 octobre 2026 ; le journal capturé est conservé ici sans prétendre que cette copie a été exécutée séparément. Relancer la cellule reprend les checkpoints à signature identique et refait les rapports. Les parents étapes 4–7 doivent être présents dans le même Drive.

64 vues de questions bibliques, issues de 38 lignes mtBCI existantes : 48 train / 8 validation / 8 test, 40 familles de cibles. Français rédigé par assistant, réponses baoulé exactes de la source. Pas de nouveau corpus de dialogues naturels, pas de validation native. Voir PROTOCOLE.txt, LECTURES-ET-DECISIONS.txt, CREDITS.txt, rapport-etape8.json et RELECTURE-ET-COMPARAISON.json.


In [ ]:
# Mandia stage8: own weights, private compute; all full checkpoints stay in Drive.
from google.colab import drive
import pathlib,sys,subprocess,base64,gzip,hashlib,json
if not pathlib.Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
assert pathlib.Path('/content/drive/MyDrive').exists(), 'Reconnect Drive before training'
ROOT8=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape8')
ROOT8.mkdir(parents=True,exist_ok=True)
ASSETS8={'mandia_stage8.py': 'H4sIAAAAAAAC/6VZa3PbuJL9rl+Bm60tkAlES7akONLlB29GM5WqvMb2nbtVKhWLIiGJa75CULYVl//7nm6Qejj2ZKbuh8Qi0Wg0Gt19ToOvXr16X+R1VaSpjsVdEtdroe9LXSWZzmu8wPOqKjZ5jOH/SRapFr9fiDCPRRwm6ZYeKl2m4dbrfC4ws9ZVHqairHRdhUlOOnWyWtdmAjmzSWsjihzzwiUk8Qqy74s0XGCqjjZ1UuRe59WrV50kK4uqFmVYr9Nkof7PFLlah4YfKixfZGoVKbM1nWVVZDCmDqM0NEYb0UwNTZxEdauoLqpo3T5kUJCEganDlR5AUJjBc0NDHho+NzTiodFzQ2956K21qx3Q307xrzXtE7+9Or1S+Af3L5PVkTgOgDzcSONQdB4sl3nn8suXa98MPPrrlWGFEzqRdk43KrIyjOquOTVdXYelPpedq+vp1yv18dIf9Hqqr7vDTifWS/J67JTuGGe0qXKRJqZ2srB0yMleWoSxUaVHQkGNA3Vcz5RpUqc4TeO4rmuVZDhdxx13BDm9dbAXbeLQS0wQ3iI6QkSL43KwHI8ulv1RYDYlbU/HjrtT0hy39xV/HXkSITJpj3GV3OqTT9tf6K90PX0Pk2GLkvwGG/q2SSrE2kIvi0pjxa3IilingnZzwpGY5CvZEbG+TSLtW3PsgyPJLOlO4Nhfpn98eD/17cDEShldB/kmC+o1ucQ4A3fC/s9u4qRy2JSguPGvq43GRuriRhVpTKeEHKAzou1xrvjmrafzCHY5fAJ4JEUnstZhtNZVt82z7rfQo7NIpaugD/MXlHhP59vJPIQZ3TJJi1o/mVhWSVENVeCb4fHaw2btb5uiTuJE593G3OcUjEjB6FjBqFGA2EuL1UablxRwmPpPNxss8Tpan/AxeWUtdzEAz5l16Nh5ru9LPVrq4eJtv/f23fngND4dDQbxIjwdRdEyinpn0fDdMl7GvfCdHkTxYth7Fy7DuL9cjvpni/PzoZZkxJYioTl3+tnoV4j7IC2ikCqPL6NyI1VTsAKqU+2x7uKT9cwk9qbz5Dv2AVtPhyM59/3G8CY/T/Yy7BEJLfvC6j9ILhZyfK6kNaXVND52gPpxrfGfmKE6QkC1Lk2AtYKwyuTYlgGZVnL88VJJo3Usx4NTJe8wvCnxu6dkUdZJRqrkWOJszFpcxGH2bwEtAmIKuVWHjvdOee/OXSXuYq/XV8JkRVGvvd5QiShNyr7k9SFv5BhbzMI0DfpnsOHtCBulShac9vDYH56NHmFKVJTwgeTYbdAHVYNPA8rDTHNl5xKCp5IwivcMOKGCt6m1+PzlGsWoRlDFdvVc3wU0C3pHA3Gb6DujxKAn6rBa6VoswyxJE413Z+eC0xelQfxWmBL1whSbKkJBKWhOkiNQE1QoEa3DkvCKC+FE5IXAIha9LAzussCaACldhaiu2NxMruu6NOOTE0KovF4XkNx6RbU6Oe2dDrxlgvqdr0wXw95Z70Sq/YTqPrllyXBhTvrDfh+eHg36hyLP6Bwiwu9whHladvve6KSJiiy5h0u806HF8d8vzpXwzoZ2B79f9Ed47A8FqtfVpy8fCdW35/bVIsQikY4xRG96zRzU1dzYw2peQ4DnDawbjE51RMO07lmzrrgN0yTmWeL9VLw5tOHHsVPSuTOgedkXZOGxMPGLGnW+P2ALJjTv/fSfPip/jcN5wzFK045eIoSbpY9fDycCSaC/awRlnuPkG2SptambKMPCtzrY2yDHv4ap0UquUETGB4CHoAssogQ5ItrpcU5jGKleV46VDIJbXRmoCQKX9VME7xL+gQQJk1O7al04BxzAdXc1wxUwU5QIXTHbFSJIIQ91+qRCq50AeSz5oYLvx8mWk7vwPky7i7DYAG+IGOzEfoYl6qdgof4CFqo/w7v5IzuN8Ok/c9oRAwmCZZJqOhG79hE988qtVEfi2MRuxvwN06rj8dF+vLHCW6XFwpGvSZnrPj52CP7CusiSKKCdNSA//d+v08sPn6afry2WqD2SELsAHWswhOGUaZnI/B3NdHY003n9egceEb+Rc9edZIyJxF9rHRBtdnZS+3eQnDSUMeuI/xK/btJUtCzf0i0cXXSDwk1ni2mCSW29RkFlyi7ef/2XWCb3VBg9bJXgwT+23MMp2WQBzUNyOe6EUMPfkWCHZz0jB5BWC7UlqjLwFgQIDscKUJJslPPZeDBXzZyO7Wxs7iX5UhMb0WyE9d53n9dxoBMW8PL2N6ysQ38J59SO871753qozbA7C+9BjhvOSGUCNd2ztCGI0sJo57u6Uzja1B+Ciauq+TWALdYYmM2Elyj3+PviyIDF3gKIs5tBIRrnBy1jWZZnpwEMCWBSkQIcA11VBRCdjVaSx9kYOSZioyQz8edmtBtcdO8Wx1tsJgFMad6yCrnCB6DgYXowz7MCTrcPDnd3+Oh6VggKdYgewn2kaPqtCql01HZzRmQbU4tqk4tiUxvsXRyfEsKHBWfE2YImpsDCgsB2TUHgHKcr2gZU2GS5dZilP8/sX0i/P5B8v354f3H94cvn7vTjxeVvH66upgfpyKZwJqY2qhUdl6IT/75Qd4vJKsLZpoSGThsjjA06K+ttEFHNI7uAWWlFVPvj5YT/9z9ewtpqO/65XZfTr5cfrqaNRZjdbhd6wTFgFLSwTl7Flo0GvrSTqUVoNIf+NyLr30LQYmPovU2iA6DbZdFMkoycT+LjKQypP5lSUJpS5m7s8jDqpRk7GhGAL9IpT0ryESnStJztUX6yXkUtUKyzwq5F3clOpjefSX2PQA6YSMo5vNBUugcLOEGElPmmJO+MH2JQ5jTmnwWIbFak/LtUjKbb/iBgjXJcqR82MAblef3tDf0fv/FOXxdgHK8xU6fJilbDJN/vM+t16FhEYsTnItcCMOUU//Tp3axdfk7UhmXLdqS1BkNWS9yO7OynoSGnHoUBuG9knISPH3UaVcWHRxBA2zgEyklTLGs0Cxo/V0yU8SMFd46Q8vNZ8t+DA4/NnpRdda7AORWTI1c5TWy0g2CdcMNutD3KtmRbZ1ol/b0YB0srYw1+KmI73wNNLVvVVlvvBW27FQcHMtge9TT+A/xFXIF4nAIYEWXYQ/HMNj5zD+w/My0KH+MbXiVLgan/8NESjbMDYMtIIwnY+oDqhYoaULfmDE6PqsbBUEAYQcMWYb1WnJfODlByEi1Xvr0TczIPD+7EJKucmxS/uVnzLHWyF0HxJiuN8yD3+5Pj/W8lubUkRzwqQwX3Rm+NbZfbiwJU+rW+j5MV0JB3/nx54/rWQBkN8k94g4NStRXKUjB2/M7uJoFVz1Pc8vq20+XA9Yc9WvLZyndQ5dmrzk43OF57IaEeWoY0hrfUIREaw897qvR3W/RHsosiZUYrzlscpwuq1g18J4BBdJFopM0mA48FUlEA7blq5u2FLIo+i152W6y4i9jGUTUQsTeBWY3I/ipKNZchqc6dh9vZ3npOHsSl7dDlnE29JVNpKY9qPd0gPoIQ9P9Ui9kgzG4Tg/D+a7qiNThN7meIGxpWiEU/DbNFHIqbMe/zZn68xAIx2cLJS7B/Nf04fc/YevHHxefr7vX0qgX8B2n7QsQGL81XGWi50XXcjG9fXoq3caP2G2lKxSNiiMTyog42hu5nKJMeG3JPQ0iINFzolKuKvUpHOZqs4B0uS1az3irWyL2MY7FLthDOzescBc/CgGxxunk/J82N6hk0UWQSpB1BO2t/Cq6Pk4NZMwmb6PbD4jAZaLVRu0VeCPbjO4V7NFbvRu5uhxbk7D29nD/HGKztrUGTH6YNadrQozuhZgc7YPjJTLrTe45lvDCvydz+wC74ItN4IUuXB14z3Qc+6scm2MiFu9bLrtf5AVgOS729rPCbsGnuF+XfzPIW59pQpegoD29Qny2af+k+dfJib3rYlR7u6JkG9WlrSpsrjypre1atI+jl33SCLZj+yyhoXS3H9q9qrj65CDW0D0VssT26J9qVDNgTcMWEM4Ifil7ABD5piwEYV5IltaEbRTSX2/3XOL4J2d1KMjTuLibhTrpYsnEF1kYskjiCgngtuGkTv3788pXGrML2rnON2KXeS6zDWy0WWoPqNMkX0/e926TYGKxAcXJwEzgRBu5MtBFRiOYNi6dhCeX2vqy1LtfGsAX8yQ57ZvOXKOp5tCWaq5fLJEr4iR2D3EtqbCcNkwwF54WKXYX8Tan5ANakjz1D7oJSbfxZuYdQvg+qmjsY6RI1K+kDFUk2X65kEJRbjocgkGwy5pUEvXANCdAroDPOErwnNtTKO5LHpdVgB//hy6uLf/0x/e3i8pdp13Zu06Zbm78EQC9NAP60wYG0QgnY5BRn/J2sjRbeLYLlQdIllL1r/PE2jPv5bU2SJZMa+twHuAJjQYA++SLhPu5dZ+y3O17FZU6DApnXjvx08fmXDxfB9Pri6/Q8uJ5efvrweSrVEae0GbDPhOMsmjW4NW/TCZj6cMSIgLAHj5CzExh5OeUPVFisOxyyb+aPL4Lxo6sQimbdfv3ZlV4KpE4HURLwVW4Q+D7ig9oPBMfYfhLt/D8OU+0KUx8AAA==', 'mandia_expand.py': 'H4sIAAAAAAAC/5VXW2/bNhh996/gsgdJK8NFTpoFyTRgCJotwJYVabsXzyAoibK5SJRGSnacof99h6JkK126i9HYkvjddL7zXXp0dHTT6axVtT5ujLTSbJRekZubOyIfG6EtDkhdkJ+FzpUILKm3mrzbqh9++kCqOpclm81utW2UkTlJd+ROtnP8kfB6LTXBhSgZJbfXP92T+Ul8HlGSdi1p18oS/HuSpj6uu7bBs2wttJblTD62sndLia6dqFSGaNkZBGJkU6pMuGhheFUb1a4rgtB6UYHvjSzbHclKoSo2u6uJs2a0KH2wZCvVat3aK0iStn6QWiEEUhsiWnh1dr/ORLaWpBS7eghqJS2bHR0dzQpTVyQXrYB5ayXeoGpqA782V1k7G+7a2mRrL1v1oHEr/5jjbxT3UL6bv6MEX9e1LtRqNstlQbYql5oXhQ4bYRAOEJBbd89zVSXx69NzSqyUeXJ2Gl3OCD5ZsUq8+0GF4UnUH9VlnuBmEQz6wbJ/rIqp0W8TiHlT7mOEspL8KspOvjGmNmFwJ7c9GSDrs0KqzraANUMcLjkAT60UEA6iMaKJy2Tiqz/+krz1LJO9cibKUhrQ6v7uhytSCKRamMoSJwMoEY92dMQpUTqXjcSXbssd641tkX6PNzMAta5YUZsHbvQqzOVGZdImi2V0eD0viqx0ouQOyNB9RfvzHn/Au89QuE9Q+NVXDlkva1vRymSQZv0d73PgjxEE0aKS1BEZ10qTITlTUaZaWdlwEt6IZOL1WC5BtTUks6YLD1EigwEDpmwFY8wzOnBOnE/HZX/aNZ+eHTztX2Lhni8Xl2DBkmV1s+PhGMXBoSz3LnOU/382y1x580nkf3NL/82x/cQ6Apk6sGvRyC+SUdPfP9d4mdYfNLqbzFpw2Jd4TrwuCV4505+N+cVYRyaUtcj5JMf9JbWtwU3y3nSDWSPbzuhRy5f+RhpV7EY2eKKiBLjuKt6ujRS5DefR7GUWx8NJlhz4uqkzkXKL/pacXVDXP/CzdnaSc/qw4f5yTv8GltQZWqXh6IDS2CSmuXz24JSODen8jOambtAnExb7CMSkdrKIyY0ow+gqTQ6dTdB4fjGe9DqPiX8jz/pwsTijFxCic3qypLiJYxqf0vg1nS+XXmP3qQYkvqHxBcRjOj+h83iUnHQIpQuJIswkd6NgWndPIhHhI91FV09pkvZX+zMMjVYkBTLbhuGTOH5KIyZSlC0S8BhG0Se9pZW2RcNibj6YlmdlbSXU6FNKRVuXyVwen1MzXL0+aHv5wYj8A5mFFhNm5bwcxxEMTO4mFaIzCiRVnqTMJy58PJz2/LPJXa3lVaOk74f7U9emlCti4wog3PnqWcTTjtnDQwc7KfNUCHeoXHWpXsVLug9gEKKKHog+frxvJhrXv8Onw2E/a/Mhm5jroZfs6Rr/J2y9hX/EN2X9HAEPoWEHZ1qzYth70DcygxOODg0670JPAYYZ5AAB4PTsIqLPKdfzDRTt2QZmRt46S0X2sBUmH7iN9cMIvjKu1v7Ul55HDXMPgOH52eVyYJPtKrDJjw7auKykzDWcnGN04BdbDMTI5xvxx9mER5iq4ea7k97cxh1P4mAb1wdhLKKTp9PG9GdQqEdcyeAy8HOVAPgduX477HzDZuZQReMcdio3eRxxFfavFLRYV8I8BM/7SzCsmBJgIwxe1ivVWu6qlCNpEt4yeHUMogHIzgFOXXbop9L17eCyL8fP2vQVMup7anDPWV6JFreWF11ZDhLPzQAzLvLcaQBbnsFvpS03IKTaSK5r3Y8zh5YCU1604QHt9Uc5XsbB5QRoGqyajqdFfM5R5woo9D0JgPId+n0/B5TMvfmPsxkyzrljAudJEnC8h9KcB75Eh43yd4ttuRHtulQpXWVDLm1Xtsk4V66yutMtSPhxv6K42eC4sfjmHP3WLZfz05Oz5aH4qxcXobH7O/VotLvAkAv7J8vEUblhGFwSHb731PO5YlMmR1duHa+uVhkmKnbA/fLk414Ee2HuPfB0Ny6SfhHFculPejX39skAAXuL35DzQpVALWI9vRyEYfDrm/vbm9vr79/f/nJ3/P399Y+3799cv/9w/+YY5GYOx2GNdabYFv+/kLxF9kJ3xPKuamzoI0TnsygRLmymVHIjsKpQ1W+nyTx6Ffy2N2SU/r/q0ewvA5BBK5cNAAA=', 'bible-qa-pilote.jsonl': 'H4sIAAAAAAAC/+2dS3PjSLbf9/4U6d5oQ2qIJ4GuRUdXd99x10z3eB72jVu2o5wAElSKIMDCg2rVDUfcxQ1HzL42jtqMvXGrHWFvenPjetUMO0Jfw5/E55wE+FBBIEWpJAqTETPVEgkkMing/+M5eR5//5mMPvucfRbIIBFv+dAamd5wNBx9NmCfxXwmk8s36gDDE15g+qE35oETODYeUMwTWeKbZc5liq+UvJjiC7mYJ5f4QpTN8K3mCm/ecjoxq/JQvIk4HC9ohN+cC56GCa8i8atZ+fKrb4fG6Wj4VZbPq2LjjCpP8OizspwXn//qVxNZnlXBaZjNfrXnAGoxszII5TDPLmi9mzPKwmomUprSdzwPmbHxJhz/xghgxjgGnrh+a85zniQieVOccdNx8fQoNIM48gM+9qzQEZEIDccXoeUEcRC4rjlyHd9zzFEgYsuIfXcUcYtHvhmGpj0Kg2jjwpM8q+b11EPPc8eOG49gGBMGdwxhjUPXcl0Y3rFtGAGG98e2gIFNwzNMczxy3FFoj4PYi2k9YZaW4ofyTZzjiF/ztGCJYIlc5IJFFZvn2fxs+WMp2PLPBV/+LzFgMmGiKNnyKsxl+TmO8baCF2SWbg4CryX1MKFgc14UfAIjynII58Ob8GoBL+dZAv8tYBL1gPDbFzgmXHg2p8/+KzVDwT5nd5/ev0+/E+rS9dn3nReMcSFomS8FmybXHxLJJPv+kqczef0hZWnFUybPr9/H1fUH9s07jm/HSXXBT/DNlE2vP3y+HuhNMefpG7gDYcR/Nxowx/4P8F6Wy4lMefKmPki9f98rloKHZzAarg2H+5tcpOEZa/56LEuTywGjmxp+ZkU1h2cafm5uaAY3/QuWZmwiUpHzEt56ybMKBlsvp7yc09gXZ3CRNzyRk1RE+LTgMSkv5QKeHbGQ4gJeDi7fzHgaSQ5nxDwpBI5TlWfw0MkQDguzPILD4IAkm1Ri46iFyIv6AU6rJIFXBPwJ8RbcfA3ujoVIQQrEm0TOlDyRHjC8SpazXBQCnuwzFpI2vGAvUZhYPRaDuTG6EpMRzimW8Aur0mIuQvwlok8jFRdsLvKZLAo6qShEjp9NDBeIYSJq0JVGvinCTH1GMoW/hsRDwzM+L0XO6IABDFrCqCHeuDmoLl5RCPiDlFl++YL9OoPrJwx0Ni0SThOFDzvKqrJgPMkFjy5ZFsAkFjCyTOEZgc87qwpY7QQWTJOE+yCcFp/9p3/x9+2ab2jN15p/F83//VpWI7ExAp6FHygcmS6vmAzlJ5P3u05BK7lW8l4ruWUNjZZv72Of21ZoCdNznciwvd4ouWUdqOSW1Q8lfwUfHJtnecllCbcaWyx/KgV+CPC0wSuoiplMCvwB7tOZ4BUTeCB+6ZVpWSnlDCuZ42MFM8ZfE85gPPhIXqD2wlM+EWow+HoMR4lcJPhVGQYCwZ5JuJ/h5QVI8mk7JtZD4HQHrBAJ3sbr7+K38+HI1ncDPvstbE2d34IWshQea35x/WHGWfoOxrgE4b9+D/bIEP49vYUuDnzz8PwOvHx/ef0+TSfwz4y4EuUwLDzagByWsdfwKV5UMEO82HQOs5jChxkibHiMIDple87tMTmEt7qcVCB/b+qHCT8SDaPnBCNDw0jD6JhgBEeDKUFThKFTuAo8OfWlaQERmiSg4QpBzxVNhyxTg0qD6q8UVGaL1RRyPjZDC2jljOzIiTWoNKgeC1RfZTOaD81U6TjuVpzxAK6wvHqeWNprUWsI3ZcUHY66sQaUBtQzA5ShAaUBdTSA+pq2Y3DvpljPrlhZF+vdl0JZGsnKKXb6HNF1z+VqqGmoaajdgJo7tNr2qjxzHPjWiFuuiAPh9Adq7qFQc/sBtS+TLC8G7BWIJNwkC1Bs1M/v+Tuei/JswJIs5PB3BkMAXv41/Ag2ATyDojYY4HkJ4O6VBV9J7CuYdgR/9jZA/YHT1VH/4dkEsYJx6iGWP82UH0zNpPGE7eTTJ1zAJmvuN/M1ar5lAcp7cf1exRKcpKu5x/C40ryvP5QsAPHO1HwFQzzUoQcBvMJA1QgPMAhnlxkQIeAlEIC9hpmvju3Akt/lDHzsKeooCk2mPclkaDJpMn0SMmHgADxe682XAiwVioETFV4gWl7Bh1neLvtfPDWhHmYFmlSaVJpU9yPVeGi3ZutEfmj6sRuYVmSaPSLV+FBSjXtCqorNMnKJZct/agSoqF1m6O06EbyikOmFJPENyaN1klWLXOYkyCffFHO4k9kfucJDEYo0ymsNryQ8XjPQavKqZUk2C24JpAAFW1TNDPCGx0mk60l0UeopFrHNr/1mvybUb0D766N/+RkFH/1ftbTjC6VAIEiWFRX6ztJJ0ARwAw8u4AVJodwbkd9wYEIwgR9KyWYV57UDbn3M36KzDQ45z+vAcRwdjuAVKl/BO2BmWF4HzY54ORp8Gnx7gs/Q4NPge2TwNRtBYPpg6mhjaWUwLt+aIswEp7ea2+nx4vB+a9KQ1JDUkDxKSHpDpy2u0fA9T4Dce07kCG/cH0gemtdr9SSv9xsVDQHK/QODWx0kCmybUqEgqogbn7Nf/sefKiYK0PWU/Q2GUwRSpEMuZ8urF+xcsJkoySQqMwn/r0rMcmLnmRSnv/xzhzcTd6vwuvSYls3lOqzATz7ZNo9l1yzXIPu//xnkGcS3gAeTQiFSVOabOj5Q7yuXYCHgeEzWhSl/SeKOsRLpRVIf/5IPEAvwE8r97AROjDn7P//ILhKkxhT+wxlnk4oPOas/6VugZXT5KY9k6hpQGlB7AsrQgNKA+oSA+gOfY0yhoNi9psyPuiocuELAVj7U6XFQ6+Cpa5RplGmUPSrK/KHbYms5PB7xWFjCN8w48vz+oMw/FGV+P1D2J7hz0BFWVHAvMj7Plz8Wg7UnDgPA4e7ZiKSIlld4i7c5FX+3/Kf1mXCZqFKZUTvjPQ6exCZl9rn6hpNvfv1hWlVpi8sKpLQ+5/o96DDnqLssRvFNVbGjFBW7K4a90x/3MFfWeq71fE89N7Seaz0/QM/R20QpS6vTy9bR71AI6UGU/n7z0gzQDPirYoA9Go5bvtPHZmTbPIgcK/KCUWj2hgH26EAG2KOepN0q/bjEp6HkcCunEe4SvK1gTZiRcw6ToFRR+HYcg3CW8Pksr1AV2B/hD5iesm9xk36BEgsvziQ5bgLMFy2atNJCRUXDDZriL7inC/cnlzBUd54tK5ZX53VS62b+zw0UtWfWPvHC2lNp91zRljMrQo8QHMCDhGM49W0yP1ATZ9MMfvswvf5Q/yuZPGUwDm7OB5jSSifN4CT0CAWC4fHvhzgAvsDTdyczfLmo4MQuT5bV7cl68nlr5Gnk7Yk8QyNPI+/pkIdWSqFsDzEkT1QzKxzjxsxuRUdXiaQjouFDLlaDUoNSg/KRQGkMvRbb0LItkPc4ssNx4IVu2B9QGoeC0uhJADr5xlSANObKigpbUiAX4JbGzfw5GDXZOisUHgcQjojiA1aptlSD6BI7VcDfYFYXZ32Zpalg32fVgqq2NqFpgn0tRdXpXqwvpdx2OBRMRaznsipP1JGK9USravVK3mk522HnIOAXCabYYnptIIuKq9jsAp7sTa/gR4m5xIop33QoSnbO8bwY3YqD21HmjXcEj3/aSWlOaU7tySlDc0pz6vE5VU8lOREzddFUVfOrdXywme+04Mr6ESVNvb5ybRIdN78edpmaa5prmmu7uWYN/dbcJi5Ab00RcdP2hNsfrh1aEte2euWonOGDwFXEQ5KlEwyCTni4Wc+IYRGkt9VJnTAbieoHNl/+BDdThcWR/igxLBsEGE2KGKQ7F+zLNMqXVwMQGakOJUcdMuJcUF2ICg4FLStXjje45q2mGbaQXV1FDV3PaeW/ozqymGC700t5TAv+yGq780q36Xb9nv3yM6p/TQqAQEmxHNgWEZmyTqSFq1y/T8mrJ+FfDhD5koLupwpIRYVR2/gTkAljuTfGgoE45u/CmyXwBuYGSoANF7v8lKa5g4JHMXkNSw3LPWFpaFhqWB4VLJt+IMlGV3bV5RetIbDBUMxgXflm6UC+EOFHzHm2FH3gj0DjVeNV4/XR8WoPjVGLMRqEozECdsTdkRFYvD98tQ/lq90rviYCgxdliemm/xae8HdKmmeZRGE+B9GCh16ici/wbsKeIyuisOjkDMW/uL2sBja0z4q17G8CaTfyHmpyN42+3bNaU2jldU0kKmyTmYvRGP8RVJlycjHIg6L3Ob485aHK+FVRGbUYk5JviDPWQepKLO50YX76SWlsaGzsiw1DY0Nj4yGxkWwWi2hEOsL7sd5m2lZsCdL2VCw5dKoaMBowGjC7AeMMDaPFLrHdsR+BUIIYRq5t9KdKru0cChinL1Vy4aYolz+XWC4WQ+jgTkZPFXmYUELJG1XHpmPS70Lmt0Thrz1Tm1qeL6/4BAaFd4Z4BTmbLa/gqa77NHbVfD9sZu3OsrtOqSV1GWT2ElRXSTT5hUh6Mdb8snEb1fVXh935yuY++coHXE6LvBb5fUXe0CKvRf7uIo/+HDq/Ec85puPiKfk+3aYeQNP3noGWcC3hfZVwd2iYLd/TTTv2rAiExx6HwWgs+iPhhzYctN1eOYKKE77g6fK/c9wonouKzRN0tmSSNnZxF/oVD1Esbuw7q7Bi/Boc1/3QXy+vAvjWu7wSmJxaYAy02ouuN55BUwOev1XNZHNqz6fe3xTqzoC2jZlQpPVHA+90Ih3baluj2fZb5kf77BTvLFl6mavs1IqcOjf2qV/z6XqTmjoBDthruGdgRZLNZhyzVIEc6Sl7Kdi0KiqVrgr/gKo3nSfUfnbJU8wS3iZRd3vD3c0ujm0dmp6anvvS09D01PQ8SnpS8HM95ZsTHdxs4f78KXrAcjVNNU01TY+EpuOhYbXYotyOXG4YfjTyQZ5Ffwrf2od2VrTHPXMnbmbs8DncZPxFq1+PzQkoa3zUwOANLig2AEehDoVC1VKCz6ceg95W1+q0ONtZtSoPcUJzvNlUvttR+RRLbDUzD1rbVn0nLIhErsgBm7Z4KJdXrJANRxAv51U6Q3tWUiQBBh1s4AOjEYITTkWQ4J8O7vlGd/mmTz4tjTGNsX0xZmiMaYw9Gcbk9sbWjdFX2UYF7XbhiM+JY/danAaZBpkG2X4g84aG3VaEPgqs2HdcgFgYetaoPyA7tEei7fXKu4neO+zxJ4tQzlGhm9JFJMzLv1ArW4mliUCAvuJwy6d8+b9np9hOXhVVr1jBg4CXq+JHcGLOV1JdXKZ8gg+yuhSDJxKHFXB3o0zk3buBt0wSBljPZbcz8+kX2b4JuP/q1ih7KZS/b6pavAfX75EXiBHQ/N9wYAicd/1+Ru3fT9nfVqoC0jAScMgwrEvbnqxrVV0kGE6yLpWURhgCXvEUh1QewyKnureMSjCpbFa47DTByPNETkWnA7OrEO/RLkZTU1NzX2oampqamkdAzabjSSioUkRT7v0GTvjNGZw+L3reZ5WaopqimqLHRlFnPDScFtvTN21fuLbrjtwxN0V/ImucQ52oTk+cqL/FKjkw+eGM/IaADVVmiEJDShRvuithiaTyJ1Qal1MjlAhjDlgslz8ucvGCRauyvHVab57LBR+gmiRYLYnncHREA9xiaCZYaSJIlj/OaCIFpyYsnLyMbXNsQkty4FBnYcWnWeTN1OcHWN1HpYJ/+fm8wnwEJMYFOiz5ZY49Tqj8UV0lqZBwBaqCJDEChVyTQCe8soowUVV5iSyKGhs+7gNcqY81NY00jbR9kWZopGmkPTbSvkYFf4vKj5cme6rMsbz8rZr/tsJnFDfR6FLdCdZPT7WHW6AGmwabBtudwOaaQ8Nt2yd0bMf0rNFI+Dy2HaM3YHPNA8Hmmv0A258oVJCiQbCwrEgjgX+zAdxNWY5CjJVsYy4LTt1O4FnBWHD4Oc1kLjY7oiRiwVW2QF6ucABjFZlqdVmcCIxmgBfPMtWFsnEs1gH9IGlw+Rwj/+GkZHn1giXLv2BW+IbXcI7xJfu0MyvbF9bBvuf5UXT1QOv8DNZ4/H4C5MFYE4WOOSIGi0Ylcoo/lvBjVDX5Buc8A8iXGNYyWGUn3NZek4ZgUwx6OYHfSskiKdMNjyJ6G9eOSCxzlcH/4J9LnnSX/u3MnDjuFWkWaxbvy2JDs1iz+Lmx+A+cFrbdskwdjibkXOCkZvDIpKc9pvEdPwXNY81jzeMj5rE1NMZtOY1xYFiCG6EpHO6a/SmR5h7a/8btSf+bdbMWUcCTMJvzSZqlVGu4XEXHFNhdBeUD/v/ilgwOPKGOpFkdu1fK4d1ncDO/Yp9Lr8mz0QMFc74n1O5ERYRkjXsSJbRKUcgxLb0DCXYXEQ68kpZqLdX7SrWhpVpL9d2kWmDpSHhAsjQFCS1YffA6mH//Qiv3le7DpqKlXEt5v6TcHhpeW/SgG4x9Jx6Z3BIiHvfIC3ZoexO3J+1N1tv5Sb31r9KI6/LAFVr4+Dt2w/gT3GZ4GEgtfKcF3a1V8vZuWJFcZRuD/jaxCXXBjP/3D/+FLzKZ15ft6gj5QLO8uXtzwPQ+CnOoK0LJWnepgYhs+of8cdUeZHqRcF6XfKq1mV1cv+9sWOK6u0MZHuTyGg4aDvvCwdBw0HB4CDhsNZLCseouUrUgf3zt7li6T4OIwyepQaFB8dcKCmdo+C1WROSFI8cOxpyPAyHCoD+gOLS9iduT9ibfcVm09R38MknQC8Nxe5c6stdlagva8SVPC3zFLmRanLJX8H0cfoabjdpE5bjrO4SPCp40uOMGLOQ5C09AaFS7kVAknCKez6lNCQwk0qodNv8ajn9bZXK1Lywq1akQZpXfPqUO2BzRejeBda+FbhV4utkHMWVYKEkgCigztnE8waslT4fqP+wCt5cVO1SvxCmvN6HxlAsp6sbtkhWUXFtK6sIC4AGZH+6AnGFY3dWfnnTOmoyajPuS0dBk1GQ8CjK29IbHfZLtPRRRj/uWzCGWUyzX84Pjfdeq+aj5qPn4ifnoDs1Ri+UoTNcaj91gbHggt+Nxf/h4aF8Ytyd9Yb4pAQgpSrqyU+pCRiXcURQZ/Gt4jJPllcBauTUNKP21Qceq3FETSpwvryZSxQLAGXB0QTHLeKwo5nDLFyBw1QKEfZ+8pvbJdMDvKdfTlZzUtZA12f5uowcKHQM0uP6QUhJsHT1MDkJVn6gBwd61jZreJnQ6wocXcBqrikq9hbHIxcoV2VkK2LC68nyPcR2agJqA+xLQ0ATUBHwaAn6VzWaqFD1TTIuoRiD5Eal0fQgqIkugykZgMzwez4iND7NETU1NTU3NI6HmeGgaLXaj61pW5I4Db+waoPg9Ck05tESU25MSUf8mZSDFcJtUP7CFVPHdSqkHG2mi8NBncEAkFhz1vpKICooFEfgQc2q2yeFBQXmPJFb0o3AIycoK8bCA4Qf48xyvM1OlenPB5lXeEfRIYwtKpG2muA4vvB2Px7CkjyIkd69lI9Alm2YXgnqpUIqoCj+n6knAAEWNrMzoxSCj34kP+Duv0nRRESnUG1vxKX/HQ/jWcJkRNlLKKL3APi0XUrALIWW6wzPqdMXHPPmsNeM04/ZlnKEZpxn3tIzbMJ9W0yuyJJEhPDNDsqJuJE3tLO97bOi73xI1ETURNREfgYje0DRbrD7LD0zfCALftMLAdKL+EPHQjjFuvzrGxFUJt8gM/qTooJvDI7C8UnEbQAPyHi6vSmyqUtbFh9IBOfuyCm7fFTRAsRP4LFCXXok1HwqKO6nyf9kJv3omIPvzLI0QCKrsQ2MwRdWaHDtbqj3WetoQd+eFbMXBXGarwregQeni+j2pfinTdFIToajLD6n0gxYofHfCMWxkQFAgttyJZf64O+zlUaeowaXBtS+4DA0uDa5HA5eqcTGB1wSFOb69kdzWKKTKb1M2Dj92iD3IojTQNNA00O4BNFA/f2haLZaYbTmma/qhbcciCEduT4CGC74VaGV5JkXFrNughic/CdR8c2yNLE8ILxz7wTh2YtNzQ2FHMQ8izgMjHhnj0I8cJ4y8MDQM344Dz48D1/LNiO/IbQDxrm2IOto/DAXcn3XpWLwTsaIrxX/UVMD8a/zYTtlXPMfQ/wQsj7A2S1K8+SKxoIyBECP0EyzFgYnb1AtFsK/xU66tlNO722qv4C9/hwJTR7XWve242xe5ht6qdG+ro05F+ktVV4oC/VXdV0kJ4vCewKMKXieOp5O6Luw6KmSaVMiFkx3Q66p59dhT1NDT0NsXeoaGnobeEUHv91uZa80uVZamtfnTpLcpNvB6ws8QfPdYqIafhp+G371cmB6Ye22ty0zH9+3AHnlO6PrBaIt9C/hDRzSVZ+rG9G4r/PvbqtOL6T1J3V/HA7IFnjGKY9P3I9v3A8e1YdzQiYQThfE4MoUlLJ8Lw/fcKAoD+Mvx2B8Z3igcmy0tU5pI/RifZ9DlgsUyWv5ILrdcqp6Vw0hQZw/cRjo5E1WuenIkJ3BDBfnynxMK0T8BURHp7Tl4Wari+Ffj03AwleVVLjGpfC7SCMNAUOGb0dT4XTT7BCv4KOvu/lPfrh0Ggv3Lz5c5B6Ep4H+Yjk0RFx+V7Mqwf2VBnS1Vl0uMythU+mawVPI6pxuocTuWPG9HWbFHmJmmkabRHjRyhm3h/3EQRYJ7NphhoReNw57RyDmQRk4/aPR7Kr74GvQB7ji1VySp3CNtRWE1xiBZXlFReLARaFcqROu7FF15b6vxwNDhE5UUBvfxIrukoMMTnnb6Ce8xqTav3v6z2WbGaz6F07aKPkrGqaAjCCyIdTrh9BbW6iAxntaHdmSPuaMdPLjnVbXWa63fQ+vdYVvQnz0CqQ9AYJyRG9qx2TOtdw/UercfWk9eqFruVC1f8j99f8IlbWxgSHc+WMmlKhBFziMOeolhBxUrqR0gVgX+nC3/nMiCB6I8G+DrsQC5HWAlYHIgwTfaKoXv71ThV2DEeIp6nMN10Jl0emu1+5Ukw1f+lXRTu0KsO4ViX6wmgQzAFzB/OY15Z3Hj41r+x2X0H2Dd2+nUPH0H1sS2z0vRBdRn+V/TFT2welXt+CJXGNaw4pScPE3w7JWNUbc7LGUiTxlGUUj2TSLf8eD6Q4lvXHDgFsCFXYKhonocYmyEGh4skxK7rkhsnIgvv+ZpV5q1Y+1Is34u69NI1kjeA8njYVv0R+DG4Yib3tgSfhTZvGdIHh+I5HFPnIEVbfgQFuCpibM0Wl6R0UIus+oHFPpA8CqEWaDiT9DNVZApc55VMkc/Ga9AM85rInCWAucKfPhOby9WucjgdUBUsnE4mVGi3A3STzvplnKT+852KwiRtmRAsM9RnNFWQoEGY5EH2CV5qpxo0wwFnYws9LjBlEi58R08/Zef4ewORo1H3VGGDzsHzRHNkd0c8Y2h3cIRrI1sGcIVHuqdEfWLI75xGEd8o0em3cpoiFSqrzIn8Ftx4wkreJVzOTxXcQFU8BfL/y7gxS8otmBV+lfWMj6rDRtVM5iOXv63yfJK7OgSsJ6LijtQ4QRNzEIdWQCSHuaZzEnJAS08TTOU9h023GOvs7U7wP0XuKZV7fKrCo5t48mqqc2ZLwAiv/yMwQfACgQI8gB3hFhBtYJhpdM5H/Iv2Jdo3cwwIh2Ons4xbH0CZkuquDNDfyEYSVVRgR1zwuGjA3uoUgWIq2pXonPXHtbRTF7jUeNxDzyaQ6fN8xm5IuRu6NmGEcSjnu1y+eaBeDT7gcdvyk3H31b03auaBL/mAah58gLRABgoJdosdZUMcgI2oXh458mccrFAscD+WGSXTZJWiY7EHPvEAFLKEz6fZzne+Er9A/QLYrzeAoPuOutGqskWaP0sr7BTWs0XNHh+gNsSzlMvgOnUPJmdNSOPb/mtpUIOXvcapl+2eAXZd/V+GRhVcVIjhZZ8/SHBKovD6/cAm3Q7qE+SPUaOQGWT4S4bnjkllpEzkEIF8T836vrHtZuRXIzU/K2roKTfwddjXo9GrkbuHsh1h25bJzvOg5HpjLkF8m0Gbs+Qe+Bmo+/2MrBEVCVDrcAe0GAAzDGML4tUTCDIOZIGjDkFl6ZYMfBkNkdHGRyQixJmnXK8z94hxjobD6wuq3qL4jYZDR2ifVcPSzZeiU9Z2tk2+5MvpLXjwIEr2A5gkSxUpYfPq0TWe1tUmBjsp4sk60CSZe6IUtlnaE0HTYc96OAPx20Gme2MjRF3HM+0grFv9IwO/oF08PtBh9/Cl3z5g1z+OMPcJlkMlJFB1gRvzIDaJlEqWqWCWpzRxk9TRH7AvufveI4hGC04+B1I8mpYWRsOzaC4/7Q5gw4CPPxkNyX/gFluq/w6QgFjBt81NQTTSQJaUdA3f4oE37AE1l//efpubThgIOEkwLIXdeV66mimyhe+X9kaUq0DLoh2SWelfHMXRo5h7hpTGlO7E5UtvzVXC7OULdewIseybS+6UW0K7qBnmqZs7UxTNm9PU7aehFKWNeJuHBux5znx2I58jzvcM8OxK8DKNEzH8oQTcNMUDg9Hhsktzzdcbzzmvm2AIXprwamUv63UTtNLEG+U8hl99eel2v9hX/OFjAYYFPGqikjpaVcK/Wy4/4Smwb8COwHNhOVViV/v80yesi/n+fLHYiv6YUDZVAVHvxuW7Z1jRVyQC6rNhG/B+RmoG7y3kGmOPrPo5HcwDUwO/jaBOz3P5QIjBFf1f3PcmRGzNnvpD5zWSxNcrXX5P0t0t+FrNJioM7HqSd2shHG6q6RVPz69TWI/2Me25vhLAQBcV6R6CYBKxPWH2YC9xlLMdQeZtEpP2UtqBUNtQqkPDMoRcvGb/Pp9RE5B3FmDjzZdoLsWW5Iqf+Ap+44azGCmMm7B0Q7ahRQqzQx34C4kxyhJzITGl7D5KHsNWvyOwzVnmHjWGa/ZBftntUD9jUB/I9j9jcD2WvPlglFsx0bohp4Px9xsMvd8vxHY3j2+EdheP74RICTiivigyjSCsJ+gtVcSjCY5lfs9zySVzwfMNUlmy7/c5rCkqoqYXrDNi80EtRpbCJblT2V3f7j7zPCmJ/LwqW1bqIFo0tlop+ldzvkMLTUK5jjHTGeQ+Sbe8ZzciJjrNp3zDt44/g7b8p5X1RDQENgDAn5rIt3I4iHnBtiEbsitkdUbCNzHLLT9/kAAlp3XhoLa5KFqvIXqo7mQuYpmqIPT+UKEFLGAZwzYd7ixcxsO0Af4sd7i5fjH1+rGwP3neBMId5/ctomToAmAmVMJBsSvBTrgU07f/6nfJf6L1+8S/9EOW+OAK2nB14K/W/CdUWua1tiNIs+whOeODSFMuy+C74zuIfjOqD+CTx20GsdQ3USLgt54lOVwbzZC+oLNMZ4O836zapHXKisqeDICPgEpLZoy7Vkc12+jayrkkeDVD59TV+aTLB+oH+rCRipPCrX1Ms/PBOtuOr1WZ2zOBa8NcTpn8D5X6cUr1e8GyJGtub252B0XuwWkrQ5cgXJDzZqK713hEUY3frrHZXBECvZTei7hoABNM2zmXClzJGzKz8spB1tlekHtwtR+FxaIolALFbmXs0cHFz4YclKBXr6pHz38aDS9ng+92pLDvMAexVEQeP545Fu+ppem12PTS3Uhqceh0W/KO1cbPhyMkBxnBw/ijak9W5o92OK36PaQjGnhoGUOGNxxmoSahM+RhEZrHphpj3177DhRHJqGHTi9IaFxHxIa/SDhN2lRwY1Yx++hvHIUUlk2UXssX/60oKCEVJWEqqO14Z7llVLg6KQORnhBhJjXAIBHl54ZfOQw3fdMYGIN4QXfR77UweBYu5ZfthbtXSUXbyo/3Ndp2pg1nRfoSAA7poW3plPfY8Vr5n0LqKDCGTFnoBgpq+K6WC7t7tT4UVEFVA1qI6pQqAMSTsEBGENAu0ZcVXgKqEgJ/iubQATVyZlCDLuiIoyuKlZPOmHt79Sc3IOTdmvylhs7jgvKHGCLTR66veGkfR9O2j2Je8wKMT9DgykRC471DMMBCn69S4Smz8qpRkV4m10juCvniZCY5lvhnYtvUtmnuJpITBZmyz9PLuel6Mzeqi/fVHjHqk8YQVibQMmJYhc8V+QB7Oip+emX0Zq7dcD8tws3vs7eXX+I2Tkv0HaaXr9PJ9X1h2HdozLGqLWP9rEGJP3BCcfkXPbNOzkvu6oQ+ztqKz7kFDRoNGj2AI3TmgcGSmZHHHTR88eGPeqPa9K5D2icvrgm2SU+DCVn51XxtjrB7RqQ4Swv1/ZGGyteNUdjs2K2vFosr1Ilp7X44qBN7YiVXHc6CPeZyabc32cKW4UK8YFCDcXv5vB3xC/u66/siTw0S3fPkbU4a3G+Kc7/HxVowLQerQEA', 'AUDIT-BIBLE-QA.json': 'H4sIAAAAAAAC/5WU3XKbMBCF7/MUHl/nQggkUJ+hfYJMR7OSlqCpghwh7LSZvHsXjIntOp7pHft3+I7E8v6w2WxTPAzbbxtZPU6RjWOfp/idIopzAt9TWDWPx8QegneQfZyyp2TGIU8hRR+zzNj71xF1hvSMWbfw4oPHSbZi5/WY/LPvIegFomyOEB3sMqYbGE9zSIkfkOym2D5+xjl3HsdNuZ1TP2/hrtPfx2n4onGx8PSPIF/6Vm8QgoZ+OBCgxjewWQ9xTBb1sKM0aeQ04ty52lsMUUfwedC7hAOmPbrz5j6mF4L9g25R13GPKcBOG8wHxH6ZppkWwrAMkbU96guXZ1U8hN/axhDQ5kl3zB322VvtwFPFeQjxeZwv5ngvxpuA+hW0H3Qf81XfOS71TEc2N+JBm5Ee+w6Tn15E19UPYQbSXQwujuS6AyI1kxEk3hGmRoNtTBe6ZAP00AEXktLbRjYIpQKuZGFLIxTjkpkS6haMKERTmIYDutqZWtRc2gpMXUrlCnQogeH8hWxPF7TKOstN65SBuimtoF5bCIW2FKY1RkrOpFCN4Iz4yqJVkjkowSluLa+YNe4om+Mv7OnG0pmyNU3FlbMt0ThgTDilmopQBBc0X9eFFbxgUEmjFFaGcVZVsqlbVfPaOjgqY2+jo/MZMtzaxvfTV7rszWk7P63OaFOJq6Zci8s+rkU6wGotvsCbvp5u6ovy9Xyl5urHrWW7ZvwaUVT8S0LeyPuA6i6gkBeAy5L/B5oUX6MxfhetYMV9tvLItv5Y/hHRtBx650OcoAvJHj4e/gJFRPyCswUAAA==', 'LECTURES-ET-DECISIONS.txt': 'H4sIAAAAAAAC/5VZTXPbRhK961fMMalAFElRsiSfZPkjrqVkl2VnU3sbAkNyEgCDYABa9imnrdrr7l72ltta2ao97S238J/4l+x7PQOA8kc2W+UPEgRmerpfv37duNRlZvWFKyqdNur9j39XuUmbtjYqtYt8e2uUaVS2vU2tt670ytxU29vaFqZsdG783mSmXNq4BR6YjqfHe3vbvzS6UWdq3dpGNeamMV7Zstnermrd3ih8wKrpWter7a1Xmala67FnW3vlXVunuL3C+trW+HRfeVgjG2dGFdvbZu0yk6gaz7Y5NvI0L7eF5TaVqRtbwjJDO3VhS9jvR+rCqMylLW1WfntbaRgL23dXyfBVt42YgZ1K59X6TeWa9fad5xLnbdqWSmP9NMfPBja6dmNUi48aduXW1ZqGqgqHwOpqoV2LkyrXhkula7Bt4bLtu9yM9vYmI3VtyrK26Zpn+NNal6tEnV/M4cfJqUTihdnA6zjSSs3d6/0XJjhIXcFMnatLna5xXPWy1qXPZfczda4utDfqummzN3vrpqn82cGBTnNdwnW5W70ZuXp18Hxyuj8ZTycHe/MY7bPB04fvf/zbUaIavciNbtWU5umyNDdGnY/UPDhulesVPmXvf/wHHOPf+Gb7rjCKxwFkSji8FO9jWdXQwOX2Xc0AFLB8+xMch8twLoLWACipq6sWfn6IOyMaItJMSUi8uLpK1MaletHmhAZNyICwJDyPZ1zAAk3DLbX1IRwp/cp9l65ujEAgN0PksWOHbfiTqQBPpPCCqTemjsHUvIIwtGFJPN+4701p3+KO2fj0GKZa38DDwCvOZbzXFlGcnfCHlak1DsjrJRyx/XcZrMjwiC1TnEBvTLqzoq6Afx4/zVtvN+FuDzvwtC2BZuZqjRuz2sG+Ro2TycE4meLvoWQDPVEYPMEEQyhkaS9LbHRuMzlFojxdnbrKYqnGtb5LiRhZxAgHXzrrAdYpMqDWemG5waUr3ybq4tn86dUTJv1YwPqsapCGbwlWAST8XWBl/Pd7wft5uHKTUepyLL5f0AmH49knoTsFdA976N6oCb4fB8x22Q1yaOhR5KMuAh7ARfhJPX4MjDXbn3FHwvxeujLjdRy5YigCOQSnA6pzQN8sl6THVj14/ggRXQrISSCpyVur3II4AglEUhAc82dJ8fsfeNzRg7S6x6wgA3DbuBzstYtVgpTWpwSprgMPLYIflfWOzD2Cr/EjtjAVUi+DZXCCK3DTD22XCoLvhcmVLxzojuEDm5lNYC75/bWrsx5sZ3eJDCy5qC0pmZTJkmEIqrYWDm7kaukKcrqryOidk5gB5hOZ3hNMx8xVu8gtkeydPIMryBwJkQ68DVDLMsF/NEDcHdIZ4D0E09pV6WpXCn51PkqISUDyav6cCD4SBD9aLm1qadl5DXA2AV5ePf4Qwv8vdo9GuXuNlcq82p+Mjn8H6d6oaaJOcOl0p76NFGprmwm+IuvBZY4Y9fss0eGjJPoP4Kqa0WcZTSK9CvKYEkdj9b2q1jUqBX4kdmfq+x7RNtWpRZ1m2a+AGwkUObBY2FIDXOVOdhzE7GHcsO/zp4+uQYDCd1IQiU4UyHewxh942MFT25wlVDgKp4GJaaRmBAj+JBASft7elqZe2Q+Qb8rarIABcmC6rh9/9VWiTF13tduD7JLdbZPIgPjfFJVXT56/SqglCsdCQv+GLbsqvwLDSGHb9UXJhWtNn7DuMOe9aXOCexFSfmdHgG42lPdEfa2zzL3mVg9szQuxzB8L7p4WcOeGmfd5bgTpZib36rVt1iRg4cIW9z7Ujf6tOn+MOj8+/TTkRsTC4QhQm42moyPah0InZceh0LyA3KqR7Dpr5Qk8G/Uc0pPGZFKsNf7AliAYEbeYJJB3EAC8MxJUTxm2RGn1uFOkH8UT6a4nO/oxz+VO2edD2utqM1M/Ek6ntXRDdg+iL3IfqqnJcfev/+Jqd8/z6y80y9uiygM1dumUCJ/SkJAkaolIbP8J7BuyIrULHdHvyxNR4iIJdK9SQbvgKdZRKTmSk0imjx2h2sbm0Cs4SsSY9RW4Ct8BorbGZ6COaM/JuzH25ADwsV4uNe60IlNiwmqAOdbvGIzBfkDzaKT+YAtG+0XrAcZHl4EGIxyvDaoDVtifmw3qwh9K9zo3GQL0kIol/19895CQmxyefrY+g+Xc4jteWbIq40QNbYn2bm/D7uLKbGfHeNyqK2VBQ7KbeLeRb6xyDrkZxf0Q516lkrUgzshTIDPUHBEC95XJg6ZfIeAl1WioLGTNmvpN3X1WHG4+if67WNWkE7YKYSdeJHvfORRSA64Dl0MHtjc4e86CCa4uYbPe0HqBcZpSLKM+jZj+wPJ/iFk4GCEi32ZBEou5lMEm7xyBNX1ww4CCocxWzKchg+qojqgLRb6TUGNbhjKPQyxt31TBxRsNLvc0lXkmabSS3lBOp723ko9J0AJlbAXQp0k/J6XaZp5JlQm2UTRYYKJGRt9o6pKYpYgbivdji9Jernzk0elMgPvt/tMSVSEE/TcL8my0jCvs42eoyd8qyQwfqSDGz6sVdTjLSo3ks9FUykDLdLasGY0Q6K7sJG/BNvg+qhg72CoqPcJI2hQho7aOQY0hamzdRSSW8/4psKizxLmL5Df/uIMKJHNX+c7n+vJ8f6ruPTiYHD4IQTJkEUSiEY1G7OU7nEZtwWyQpq3jsQge1M9MVzH0fTP1Sfihn0zjaULDJyj3IoMH/vc7TOiT3hVDa43shCLJhZRByNJoSau2akXBg5WFC3dmBLF+RDcJ4eQh4LvimG6oIRZynYYywjwemiclDMw8lntxbUOQ3kNWrA1aqyfOZUukQ6j51+tc1MfTi/mLgWavTDPF3wGn9Y3dCED1wh9MjiaT0fjoeDbZQSbSgQwXrMXCEalqGqDWdG2XDv4mcHlWLT0OkFWmkUjRmTWS5iRGDTwjVhJXeQKcDAh8B0rtOtYPYnt3BjSEmpUccq1W16/tk/krde/45P2f/zo5OjwmVEkpYgqjoEuK3K5vllYBt9jSNlazFpZSEN5iGycPlJy25KG5yMkPkZnqhuoQXES6JA9CNTJW3f1K70j5HtnCSPHOWDbSjp6l5dH5CuBq1oXpcjCPWS25W+IeVxJBzwfRJzWlYadypiaH6t7JVB1NTxQ8gKCrk+OxmiH096VNYdDQw+LO6T11enii8GWkvqFXO/pQF89fSbrdzTUN4OtGlOtZPP8QackoCg72f7VAv2wLrgqcogvQHM6wqhnOHRD3jO2OF2Yu23wnGz/2N84qVB203wFENCP+k9oEoynI+PCFQzsJLccI3hl4ID9ORuprt1wWuix7KgcbTyUfXpJVKYI53wBp7j8L7RySFA0G/qXexYcohL+4gERGaFFIv/xMDk2nY0jco6Oj450cikzGcmDhEo4iwzk75bhDFskOVbJNB+IB0aVwKgIjCgTSA6wE996piJTRrFyiZ+mxu6MfoyTnfBdVKROOWwBHnrrovvRIwDTEqfMdB8WWqi2l0qDy70wW74yofIsk31gv89soaocWUYiv3r5bEU8QVO7GFsJhYTTRDYyoTYamRiZ6YZY1G485VxmP1SWf6G6nLVP+cogf0GGB6cmjUvkshArVS8i6XgAGTmV7FZIxKjAjzYcINNG1D0RwgeSLNm8schGXWUUoeALXcZ/Ya4gzdXAznWLKFlHrpXHz0Ril4i5hLgx+COonPAZHS+JsWGJrQ25l4QkgicfmCChtujkdD+sKgIw94Hk10EZUEXh47xW+2dw14u7jGViQlYYijEKe9VIUXacYhAkrjp5FEIAscrsSAgwkHZJfht6SgUXz4OIpTjY7CXbL2DBRJ3eGfyeCdzR+Y8jZwkoGZFHEfiQO+8lLrLEyXU8bHydY6VpXNjJgaM4iy4vRIT5rEL0TGuqH6UIxrDVgfEfp80PrGktGGjp8IoMlCpFmE66zwop1nQM9leQdEuKgYi2ppq4fvwxDouikJjQOoik8+Sp0BZKKvTXdIJoeEDnnysDsUCRNcFIPSNPEAZZuM+t8L54zM7zLwN7du4cPUXbW4zfheE16ACFnap7QxEmbeSDLA08XKEndgBFZu+L8fI+GHh8gjpodaB2mlGG0Ek0JsI2iYCB234YeeohzBBSiWEqcrpzMjKQJ0S2IKa6LFohJm8tgRHxWiqGJjPrwL4gsbJfsoohtV+jXu9l3SCT0xWEpCCmsnrONelVS5lk0XGHaRVGM/Gd/JAkiM8WIQo5mWijx2kXBSp8jxXOZkpNWFqQumsmIRg1TDrJ1LhnnyVeTw+QErAarpuPkFJ/ASJmRlYvtz2xYuhcbSXcBeSm61cRb1KJFt9zsMCPTBLtvfwmcpPO0zUnRwJQgQV5oSOcfx/bdO7ecWEHgL+OkrO0Heb0uVl8QHU3wRRIL79Lmko8Eq2OD0NZf7s0F7p2ECZPB3ULh1csX50+vdpgqjFeROLrC/zP69UT0UiQcdaQOUS5S1wp99fr+oBPoXZzhrJmazaYfzmsGHCTq+J46ObnXuaznosjB4t7J0TF01KR/oSEp9pXcJ+iKs+dq57EkppzuurCCL1eiXg7DrGGi4tUfz789nwugOixZmYcFZkVCQnEsl3WcTnAUODA8OZtLtkRwIB7sXbCeRXkQXiQBqAXJzoQJlsQ/6lkScfeyMx6yO/6uTaHtz/v3SF2LFEpwSm0f4ivvoz7UwGYo0nFE1b+zEt2zvU36lkfvVLBYqmWFQXxJNW2oec7UN8/mry4f7T98dnX16NH16DsYPNr7L2y+wz1hHgAA', 'VOLUME-DONNEES.json': 'H4sIAAAAAAAC/52UTXPTMBCG7/kVmpwTYiuyPsgpR2YYDgwM3DyyvHEEiuRYMrR0+t+R3ZTSpEYNN1n77s4+frV7N0NoHtx3sPoXdKXfS1zQ+Vs0VxUnWNRqxzCtZZYVtRCcUJkVuBBYMZarAueZJLQSAkiV4YwQytlOMMxULeeLobJXroWhXNtBKzuo0aeP23cfUICbgJw1txtUwc51gAzYJuzRTpsAnbbNBsGNVAF9tlq5GpAPw61fIOuQh4O0QStUQ923G3Rw1pkY7aVBX7Zft+9jrjJ9jG4GeQdeHtpBgJTrbfAPvVW3pQ+yGdq7i9/xJlattVxCkC2QVexXGgNmeer9zTcfO/6jHv5bJ7Utf2j46eM14Wu+eAz1Vh97KEeGspW6GxWMkpcVQXYNhFGTMXauGfJL7/pOxbPp/Ulejr4NSfma5gU+TztX0TXjdNTcL14CjphGg38tb07+TZsnWdfXgxY0RZmTScRidexd0LUG+1pIfOnGc8pJwRMm/g9HC8xFihTnBE+y0lU8Gtf0Vxia5QlHpwRPrJxejyrytKkCT79ctgog1R66ZdPF+Y5jvzzKJCzOErA5T8/q1bCECZ70VfBJVr6qdGUgAi5bbVx4xVZK7KQkJSbXUxaUpSiZOD3e2Yl0EDpbPpOPxQjBU/G/lhvjnF3KHjGL9YPfl9HUvuE0n93PfgMrx6xOKAcAAA==', 'VERIFICATION-ARCHITECTURE-CPU.json': 'H4sIAAAAAAAC/22RTXLjIBCF9z4F5bVHBQIEyjYXmM2su7BoydQgcBDyz6Ry90Cc1Lhc3gH9vn7N6/cNIdvRXfKacPtCtskEG2eSXbiS199/yBwt+h0JMZOcjAtoyRnddMgLiYm8rca7fCV7DMNhNunvdlf7HRMumE64AJ6MBx8nlxc4u3yAHD0Wj6Ga5bTil342FzD7Jfo1I2BKMZWqbPqOik4IpSTvacfxF9UP7U2aCnvfajDDAS1YHMrgMJtcrguMq/f3KuM9GGurMJ4DDMV5DgskHNCdEEIM/zBFmJKxDkO+R/FScrhhP2XwrCjeS7XWQ3VODW3GMTRV19wCK5K2UT1ttRK96GQvWrG7MbdpnzO6UVzqnnKthdZKPSDsKaI1lZppKZXifds9MO0Tpm+YoILLlmnNGOWyIB9fH56OK+xH1gG+ra6ss3ywJpThihlOmNzo0N4ndDTJzJgxlVzXUBa/v0IxBOtmDIuL4X9YqtPlwrjSrWz195hM8q6GRXXZP/sZvuVU1FfVc12OdbzNx+YT/f+Mw78CAAA='}
HASHES8={'mandia_stage8.py': '00481d8e6b99c00fd9f0b6b1073262d02e88454edeca038bfc2a57d42ea8dfc0', 'mandia_expand.py': '655255a443b4a2ad7863f906603374e51bac7838c511f28b16c3acdab7ba90dd', 'bible-qa-pilote.jsonl': '868ea39a2961c3b590260b3a7fab51581b82aed7db75726c4ab7369d1ede6a0e', 'AUDIT-BIBLE-QA.json': '8d306ab3f59ce48ed7b86a4b91af82bb845d712a548e1a3f79bc9976724d743b', 'LECTURES-ET-DECISIONS.txt': 'd9a6fbcfd02de26ee5610f578aa60b56d80cc29d2a2944e9d6807082614fa413', 'VOLUME-DONNEES.json': 'bf421988daa18cfa0a4f2c0b61d7fe16adec3e3265ac4dbd8ce6139a4579a87f', 'VERIFICATION-ARCHITECTURE-CPU.json': '023413afd4cd6fde1ab55dcb19ab881316640d068713cd4ebd921e5f6b709d9b'}
FROZEN8={'mandia-compact-s2s-etape4/parallel-prepare.jsonl': 'dc2bfd9ba783c5edec159ec35bfbb66206598520bef31f960da3ad92cc240cbd', 'mandia-compact-s2s-etape4/replies-prepare.jsonl': '29334943cd507b8ffa9385b530b9625aec82c0741aeade3575a30c7df7e2caf1', 'mandia-compact-s2s-etape4/data/waxal-baoule-text.jsonl': '4f7730d32d3324d2cc819f33d7633b2d59e01f81c8e2f38651d0377050ec69d4', 'mandia-compact-s2s-etape5/quotidien-prepare.jsonl': '370d3bb631a4ad4765fbe3b15e0f6019c0470ce03c687f87ac71f6fda6b3048c', 'mandia-compact-s2s-etape6/dialogues-prepare.jsonl': '9375c6a0918a5bf8056cecd42df0c8aff90c3439220bc45bfe10e8cc700a2184', 'mandia-compact-s2s-etape7/teacher-grounded-qa.jsonl': '3cfe3c27ed8e221cad480755f9ee1a0fcee1c67583f58744cda427932e94910b', 'mandia-compact-s2s-etape7/mandia_stage7.py': '6044fb5633f0d0438000f005849f78db20c3b6c5edc2b693530818b53322ea41'}
for name,encoded in ASSETS8.items():
 data=gzip.decompress(base64.b64decode(encoded));assert hashlib.sha256(data).hexdigest()==HASHES8[name]
 dest=ROOT8/name
 if dest.exists():assert hashlib.sha256(dest.read_bytes()).hexdigest()==HASHES8[name], 'Source changed; preserve this run and use a new directory'
 else:dest.write_bytes(data)
for rel,expected in FROZEN8.items():
 assert hashlib.sha256((ROOT8.parent/rel).read_bytes()).hexdigest()==expected, 'Parent source changed: '+rel
try:import tokenizers,sacrebleu
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1','sacrebleu==2.5.1'])
for rel in ['mandia-compact-s2s-etape6/code','mandia-compact-s2s-etape7','mandia-compact-s2s-etape8']:
 sys.path.insert(0,str(ROOT8.parent/rel))
import mandia_stage8,importlib
importlib.reload(mandia_stage8)
report8=mandia_stage8.main()
